In [1]:
import pandas as pd
import os

# ====================== 路径配置 ======================
input_path = "processed_data/UserBehavior_cleaned.csv"
os.makedirs("report", exist_ok=True)

# ====================== 加载清洗后数据 ======================
df = pd.read_csv(input_path, parse_dates=['date', 'datetime'])
print(f"✅ 数据加载完成，共 {len(df):,} 行")
print(f"   时间范围：{df['datetime'].min()} 至 {df['datetime'].max()}")

# ====================== 3.1 整体核心指标 ======================
pv_total = len(df[df['behavior'] == 'pv'])
uv_total = df['user_id'].nunique()

# 漏斗：各行为人数（核心转化指标）
pv_users = df[df['behavior'] == 'pv']['user_id'].nunique()
cart_users = df[df['behavior'] == 'cart']['user_id'].nunique()
fav_users = df[df['behavior'] == 'fav']['user_id'].nunique()
buy_users = df[df['behavior'] == 'buy']['user_id'].nunique()

# 漏斗：各行为次数
pv_cnt = len(df[df['behavior'] == 'pv'])
cart_cnt = len(df[df['behavior'] == 'cart'])
fav_cnt = len(df[df['behavior'] == 'fav'])
buy_cnt_total = len(df[df['behavior'] == 'buy'])

# 跳失率
user_pv_cnt = df[df['behavior'] == 'pv']['user_id'].value_counts()
jump_user_cnt = len(user_pv_cnt[user_pv_cnt == 1])
jump_rate = jump_user_cnt / uv_total if uv_total > 0 else 0

# 转化率
conv_rate = buy_users / uv_total if uv_total > 0 else 0
cart_conv_rate = cart_users / pv_users if pv_users > 0 else 0
fav_conv_rate = fav_users / pv_users if pv_users > 0 else 0
buy_conv_rate = buy_users / pv_users if pv_users > 0 else 0

# 保存整体指标
overall = pd.DataFrame({
    '指标': [
        '总PV', '总UV', '购买人数', '购买次数',
        '跳失用户数', '跳失率', '整体转化率',
        '浏览量(次数)', '加购量(次数)', '收藏量(次数)', '购买量(次数)',
        '浏览人数', '加购人数', '收藏人数', '购买人数',
        '点击→加购转化率', '点击→收藏转化率', '点击→购买转化率'
    ],
    '数值': [
        pv_total, uv_total, buy_users, buy_cnt_total,
        jump_user_cnt, f"{jump_rate:.2%}", f"{conv_rate:.2%}",
        pv_cnt, cart_cnt, fav_cnt, buy_cnt_total,
        pv_users, cart_users, fav_users, buy_users,
        f"{cart_conv_rate:.2%}", f"{fav_conv_rate:.2%}", f"{buy_conv_rate:.2%}"
    ]
})
overall.to_csv("report/overall_metrics.csv", index=False, encoding='utf-8-sig')

# ====================== 3.2 用户行为类型分布 ======================
behavior_dist = df['behavior'].value_counts().reindex(['pv', 'cart', 'fav', 'buy']).reset_index()
behavior_dist.columns = ['行为类型', '次数']
behavior_dist['占比'] = behavior_dist['次数'] / behavior_dist['次数'].sum()
behavior_dist.to_csv("report/behavior_distribution.csv", index=False, encoding='utf-8-sig')


# ====================== 3.3 商品热度TOP20 ======================
item_pv = df[df['behavior'] == 'pv'].groupby('item_id').size().sort_values(ascending=False).head(20).reset_index()
item_pv.columns = ['商品ID', '浏览次数']
item_pv.to_csv("report/top20_item_pv.csv", index=False, encoding='utf-8-sig')

item_buy = df[df['behavior'] == 'buy'].groupby('item_id').size().sort_values(ascending=False).head(20).reset_index()
item_buy.columns = ['商品ID', '购买次数']
item_buy.to_csv("report/top20_item_buy.csv", index=False, encoding='utf-8-sig')

# ====================== 3.4 时间趋势指标 ======================
# 按日期
daily_trend = df.groupby('date').agg(
    PV=('behavior', lambda x: (x == 'pv').sum()),
    UV=('user_id', 'nunique'),
    Buy=('behavior', lambda x: (x == 'buy').sum())
).reset_index()
daily_trend['avg_pv'] = daily_trend['PV'] / daily_trend['UV']
daily_trend = daily_trend.sort_values('date')
daily_trend.to_csv("report/daily_trend.csv", index=False, encoding='utf-8-sig')

# 按小时
hour_trend = df.groupby('hour').agg(
    PV=('behavior', lambda x: (x == 'pv').sum()),
    UV=('user_id', 'nunique'),
    Buy=('behavior', lambda x: (x == 'buy').sum())
).reset_index()
hour_trend = hour_trend.sort_values('hour')
hour_trend.to_csv("report/hour_trend.csv", index=False, encoding='utf-8-sig')

# 按星期
weekday_trend = df.groupby('weekday').agg(
    PV=('behavior', lambda x: (x == 'pv').sum()),
    UV=('user_id', 'nunique'),
    Buy=('behavior', lambda x: (x == 'buy').sum())
).reset_index()
weekday_trend = weekday_trend.sort_values('weekday')
weekday_trend.to_csv("report/weekday_trend.csv", index=False, encoding='utf-8-sig')

# ====================== 3.5 指标汇总日志 ======================
with open("report/metrics_summary.md", "w", encoding="utf-8-sig") as f:
    f.write("# 电商用户行为指标汇总\n\n")
    f.write("## 核心指标\n")
    f.write(f"- **总PV**：{pv_total:,}\n")
    f.write(f"- **总UV**：{uv_total:,}\n")
    f.write(f"- **整体转化率**：{conv_rate:.2%}\n")
    f.write(f"- **跳失率**：{jump_rate:.2%}\n\n")

    f.write("## 转化漏斗（人数）\n")
    f.write(f"- 浏览人数：{pv_users:,}\n")
    f.write(f"- 加购人数：{cart_users:,}（转化率 {cart_conv_rate:.2%}）\n")
    f.write(f"- 收藏人数：{fav_users:,}（转化率 {fav_conv_rate:.2%}）\n")
    f.write(f"- 购买人数：{buy_users:,}（转化率 {buy_conv_rate:.2%}）\n\n")

    f.write("## 行为分布\n")
    for _, row in behavior_dist.iterrows():
        f.write(f"- {row['行为类型']}：{row['次数']:,} 次（{row['占比']:.2%}）\n")

    f.write("\n## 时间范围\n")
    f.write(f"- 数据时间：{df['datetime'].min()} ~ {df['datetime'].max()}\n")
    f.write(f"- 统计天数：{daily_trend.shape[0]} 天\n")

print("✅ 阶段3 所有指标计算完成，已保存至 report 文件夹")
print("\n生成文件：")
for f in os.listdir("report"):
    print(f"  - {f}")

✅ 数据加载完成，共 98,914,484 行
   时间范围：2017-11-25 00:00:00 至 2017-12-03 23:52:41
✅ 阶段3 所有指标计算完成，已保存至 report 文件夹

生成文件：
  - behavior_distribution.csv
  - daily_trend.csv
  - hour_trend.csv
  - metrics_summary.md
  - overall_metrics.csv
  - top20_item_buy.csv
  - top20_item_pv.csv
  - weekday_trend.csv
